# HW5 — 전처리 재현과 질문 셋
이름: 안세호 · 학번: 23013283

`data_analysis`에서 `.venv` 커널로 Restart → Run All을 실행합니다. 입력은 같은 폴더의 `sales_data.csv`이며, 출력은 `sales_clean.csv`입니다. 실습 노트북 `sales.ipynb`와 분리한 독립 실행 노트북입니다.

단계 ①~④는 전처리, 단계 ⑤~⑦은 각각 Q1~Q3입니다. 원본과 전처리 데이터의 날짜를 임의로 바꾸지 않습니다.

## 단계 ① — 읽기와 날짜 변환
원본의 1,030행·14열을 확인하고 날짜 세 칸을 변환합니다. 나이·만족도·도착일의 결측은 유지합니다. 원본 파일의 해시도 기록해 실행 후 보존 여부를 확인합니다.

In [1]:
import pandas as pd
from pathlib import Path
import hashlib
from IPython.display import display

source_file = Path('sales_data.csv')
source_sha256 = hashlib.sha256(source_file.read_bytes()).hexdigest()
df = pd.read_csv(source_file)
raw_shape = df.shape
missing_before = df.isna().sum()
for col in ['order_date', 'shipping_date', 'delivery_date']:
    df[col] = pd.to_datetime(df[col])
print(df.shape)
assert df.shape == (1030, 14)
display(df.head())
display(df[['customer_age', 'satisfaction', 'delivery_date']].isna().sum())

(1030, 14)


,order_date,shipping_date,delivery_date,region,category,product_id,quantity,unit_price,customer_id,customer_age,customer_gender,satisfaction,status,total_sales
0,2023-01-01,2023-01-04,2023-01-03,부산,전자제품,2772,61,34977.69,327,67.0,M,1.0,배송중,2133639.09
1,2023-01-02,2023-01-06,2023-01-04,부산,식품,3780,64,312589.27,318,34.0,M,4.0,배송완료,20005713.28
2,2023-01-03,2023-01-04,2023-01-05,부산,의류,8423,5,111813.88,297,50.0,F,4.0,배송완료,559069.40
3,2023-01-04,2023-01-07,2023-01-10,대구,식품,3871,53,297971.54,479,61.0,M,5.0,배송완료,15792491.62
4,2023-01-05,2023-01-08,2023-01-10,서울,식품,1660,55,65058.60,499,60.0,F,2.0,배송중,3578223.00


customer_age     51
satisfaction     30
delivery_date     5
dtype: int64

## 단계 ② — 완전 중복 제거
14개 컬럼 값이 모두 동일한 뒤쪽 중복 행만 제거하고 결과를 df에 다시 넣습니다. 결측이 있다는 이유로 행을 지우지 않습니다.

In [2]:
before = len(df)
duplicate_count = int(df.duplicated().sum())
df = df.drop_duplicates().copy()
print(before, '->', len(df))
print('제거한 중복:', duplicate_count)
assert before == 1030 and len(df) == 1004 and duplicate_count == 26
assert df.duplicated().sum() == 0
assert df.isna().sum().equals(missing_before)

1030 -> 1004
제거한 중복: 26


## 단계 ③ — 수량 오기 복원
수량 > 100인 22행에서 금액 ÷ 단가가 정수이며 1~100 범위인지 확인한 뒤 수량만 복원합니다. 금액을 수량에 맞춰 바꾸거나 행을 삭제하지 않습니다.

In [3]:
bad = df['quantity'] > 100
repair_count = int(bad.sum())
ratio = df.loc[bad, 'total_sales'] / df.loc[bad, 'unit_price']
assert ((ratio - ratio.round()).abs() < 1e-6).all()
assert ratio.between(1, 100).all()
df.loc[bad, 'quantity'] = ratio.round().astype(int)
diff = (df['total_sales'] - df['quantity'] * df['unit_price']).abs()
print('고친 줄 :', repair_count, ' 수량 최대 :', df['quantity'].max(), ' 총액 안 맞는 줄 :', (diff > 1).sum())
assert repair_count == 22
assert df['quantity'].max() == 99
assert (diff > 1).sum() == 0

고친 줄 : 22  수량 최대 : 99  총액 안 맞는 줄 : 0


## 단계 ④ — 날짜 오류 표시와 정리 CSV 저장
도착일이 배송일보다 빠른 행은 date_error=True로 표시합니다. 어느 날짜가 틀렸는지 알 수 없으므로 날짜는 그대로 둡니다. 결측 51·30·5개도 유지하고 원본을 덮어쓰지 않습니다.

In [4]:
df['date_error'] = df['delivery_date'] < df['shipping_date']
print('날짜 역전 :', df['date_error'].sum(), '건')
display(df['date_error'].value_counts())
df.to_csv('sales_clean.csv', index=False)
print(df.shape)
assert int(df['date_error'].sum()) == 147
assert df.shape == (1004, 15)
assert df.isna().sum().drop('date_error').equals(missing_before)
assert hashlib.sha256(source_file.read_bytes()).hexdigest() == source_sha256

날짜 역전 : 147 건


date_error
False    857
True     147
Name: count, dtype: int64

(1004, 15)


전처리 순서는 수업과 동일하게 중복 제거 → 수량 복원 → 날짜 표시입니다. 수량 복원 후 인덱스 583·1026의 값이 새로 같아지는 1건이 있으나, 강의에 없는 두 번째 중복 제거는 하지 않아 1,004행을 유지합니다. 뒤의 질문에서 필요한 제외는 분석용 표에서만 수행하며 sales_clean.csv를 다시 고치지 않습니다.